In [ ]:
import datalair
from pathlib import Path
import anndata as ad
import matplotlib.pyplot as plt
import scanpy as sc
import optuna

import single_cell_datasets
import numpy as np
import numba

storage_path = Path("/storage/halu").resolve()
assert storage_path.exists()
lair = datalair.Lair(storage_path / "lair")
lair.assert_ok_satus()

In [ ]:
ds = single_cell_datasets.SingleCellDataProcessStep07()
lair.safe_derive(ds)

In [ ]:
adata = ad.read_h5ad(lair.get_dataset_filepaths(ds)["adata.h5ad"], backed="r")
rng = np.random.default_rng(seed=0)
idx = rng.choice(adata.obs.index, size=1_000, replace=False)
bdata = adata[idx].to_memory()

In [ ]:
sc.pp.normalize_total(bdata)
sc.pp.log1p(bdata)
sc.pp.highly_variable_genes(bdata)
sc.pp.scale(bdata, zero_center=False)
sc.tl.pca(bdata, svd_solver='arpack', mask_var="highly_variable")
sc.pp.neighbors(bdata)
sc.tl.umap(bdata)

In [ ]:
fig = sc.pl.umap(bdata, color="dataset", return_fig=True)

In [ ]:
fig.savefig("/tmp/umap.svg")

In [ ]:
def construct_reference(adata, rng: np.random.Generator, n_cells, target_sum, hvg_min_mean, hvg_max_mean, hvg_min_disp, scale_max, n_neighbors, n_pcs, leiden_resolution):
    idx = rng.choice(adata.obs.index, size=n_cells, replace=True)
    bdata = adata[idx].to_memory()
    sc.pp.normalize_total(bdata, target_sum=target_sum)
    sc.pp.log1p(bdata)
    sc.pp.highly_variable_genes(bdata, min_mean=hvg_min_mean, max_mean=hvg_max_mean, min_disp=hvg_min_disp)
    sc.pp.scale(bdata, max_value=scale_max, zero_center=False)
    sc.tl.pca(bdata, svd_solver='arpack', mask_var="highly_variable", n_comps=n_pcs)
    sc.pp.neighbors(bdata, n_neighbors=n_neighbors, n_pcs=n_pcs)
    sc.tl.leiden(bdata, resolution=leiden_resolution, flavor="igraph")
    return bdata


def objective(trial):
    n_cells = trial.suggest_int("n_cells", 100, 1_000, log=True)
    target_sum = trial.suggest_float("target_sum", 1e4, 1e7, log=True)
    hvg_min_mean = trial.suggest_float("hvg_min_mean", 0.01, 0.1, log=False)
    hvg_max_mean = trial.suggest_float("hvg_max_mean", 3.0, 5.0, log=False)
    hvg_min_disp = trial.suggest_float("hvg_min_disp", 0.25, 0.5, log=False)
    scale_max = trial.suggest_float("scale_max", 5.0, 15.0, log=False)
    n_neighbors = trial.suggest_int("n_neighbors", 10, 30, log=False)
    n_pcs = trial.suggest_int("n_pcs", 30, 100, log=False)
    leiden_resolution = trial.suggest_float("leiden_resolution", 1e-2, 1e2, log=True)

    adata = ad.read_h5ad(lair.get_dataset_filepaths(ds)["adata.h5ad"], backed="r")
    rng = np.random.default_rng(seed=0)
    bdata = construct_reference(adata, rng=rng, n_cells=n_cells, target_sum=target_sum, hvg_min_mean=hvg_min_mean, hvg_max_mean=hvg_max_mean, hvg_min_disp=hvg_min_disp, scale_max=scale_max, n_neighbors=n_neighbors, n_pcs=n_pcs, leiden_resolution=leiden_resolution)

    hvg = bdata.var.highly_variable.value_counts()
    trial.set_user_attr("hvg_number", hvg.loc[True])
    stats = bdata.var[["means", "dispersions", "dispersions_norm", "mean", "std"]].describe(percentiles=np.linspace(0,1,11)).reset_index().melt(value_name="value", id_vars="index")
    stats["merged"] = stats[["variable", "index"]].agg(".".join, axis=1)
    for row in stats.iterrows():
        trial.set_user_attr(row[1]["merged"], row[1]["value"])
    trial.set_user_attr("hvg_ensembl_ids", "".join(bdata.var[bdata.var["highly_variable"]==True].index))

    return 0


n_trials = 3
numba.set_num_threads(32)
sampler = optuna.samplers.RandomSampler()
study = optuna.create_study(sampler=sampler)
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
df_results = study.trials_dataframe()
df_results.head()

In [ ]:
df_results

In [ ]:
bdata = construct_reference(adata, rng)